# 🌍 Regional Analysis: Jawa Tengah vs Jabodetabek (Data-Driven)
Notebook ini mengeksplorasi data properti Jawa Tengah dan menghitung **Regional Adjustment Factor** dengan membandingkan harga asli di Jawa Tengah dengan prediksi model Jabodetabek untuk spesifikasi rumah yang sama.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

def format_rupiah(value, _=None):
    if value >= 1e9: return f'Rp {value/1e9:.1f}M'
    elif value >= 1e6: return f'Rp {value/1e6:.0f}Jt'
    else: return f'Rp {value:,.0f}'

## 1. Exploratory Data Analysis (Jawa Tengah)

In [ ]:
# Load data Jawa Tengah
df_jateng = pd.read_csv(os.path.join(PROJECT_ROOT, 'data', 'processed', 'jateng_house_price_clean.csv'))

# Preprocessing singkat untuk EDA
df_jateng = df_jateng.dropna(subset=['land_size_m2', 'building_size_m2', 'bedrooms', 'bathrooms'])
df_jateng['carports'] = df_jateng['carports'].fillna(0)

print(f"Total listing valid Jawa Tengah: {len(df_jateng)}")
display(df_jateng.head())

In [ ]:
plt.figure(figsize=(12, 5))
top_cities = df_jateng['city'].value_counts().head(5)
sns.boxplot(data=df_jateng[df_jateng['city'].isin(top_cities.index)], x='price_in_rp', y='city')
plt.title('Distribusi Harga Rumah per Kota di Jawa Tengah (Top 5)')
plt.xlabel('Harga (Rupiah)')
plt.ylabel('Kota')
plt.xscale('log') # Pakai log scale karena variasi harga tinggi
plt.show()

## 2. Load Model Jabodetabek
Kita memuat model Random Forest yang telah dilatih menggunakan data Jabodetabek.

In [ ]:
try:
    model = joblib.load(os.path.join(PROJECT_ROOT, 'models', 'random_forest_model.pkl'))
    le = joblib.load(os.path.join(PROJECT_ROOT, 'models', 'label_encoder.pkl'))
    feature_cols = joblib.load(os.path.join(PROJECT_ROOT, 'models', 'feature_cols.pkl'))
    print('✅ Model Jabodetabek berhasil dimuat!')
except Exception as e:
    print('❌ Gagal memuat model. Pastikan notebook 03_Modeling sudah dijalankan.')
    print(e)

## 3. Estimasi "Jabodetabek-Equivalent Price"
Kita akan merekayasa fitur-fitur yang dibutuhkan model untuk rumah-rumah di Jawa Tengah.

In [ ]:
# Gunakan kota 'Bekasi' sebagai representasi median Jabodetabek
baseline_city = 'Bekasi'
if baseline_city in le.classes_:
    baseline_city_code = le.transform([baseline_city])[0]
else:
    baseline_city_code = 0

# Feature engineering untuk menyamai kebutuhan model Jabodetabek
df_sim = df_jateng.copy()
df_sim['city_encoded'] = baseline_city_code
df_sim['garages'] = 0 # Asumsi mayoritas tidak mencantumkan garasi terpisah
df_sim['floors'] = np.where(df_sim['building_size_m2'] > df_sim['land_size_m2'], 2, 1) # Asumsi lantai
df_sim['rasio_tanah_bangunan'] = df_sim['land_size_m2'] / df_sim['building_size_m2']
df_sim['total_ruangan'] = df_sim['bedrooms'] + df_sim['bathrooms']

# Memprediksi harga rumah-rumah ini JIKA mereka berada di Bekasi
X_sim = df_sim[feature_cols]
df_jateng['jabodetabek_pred_price'] = model.predict(X_sim)
df_jateng['price_difference'] = df_jateng['jabodetabek_pred_price'] - df_jateng['price_in_rp']

df_jateng[['city', 'land_size_m2', 'building_size_m2', 'price_in_rp', 'jabodetabek_pred_price']].head()

## 4. Menghitung Regional Adjustment Factor
Faktor = Harga Asli (Jateng) / Harga Estimasi (Jabodetabek)

In [ ]:
# Hitung rasio untuk setiap listing
df_jateng['regional_factor'] = df_jateng['price_in_rp'] / df_jateng['jabodetabek_pred_price']

# Buang nilai ekstrem yang tidak wajar (misal rumah sangat murah di jateng padahal spek istana, atau sebaliknya)
valid_factors = df_jateng[(df_jateng['regional_factor'] >= 0.1) & (df_jateng['regional_factor'] <= 2.0)]

# Agregasi berdasarkan Kota (Median untuk menghindari efek outlier)
factor_per_city = valid_factors.groupby('city')['regional_factor'].agg(['median', 'count']).sort_values('median')

# Filter hanya kota dengan minimal 3 data (agar representatif)
factor_per_city = factor_per_city[factor_per_city['count'] >= 3]

print("📊 Regional Adjustment Factor (Data-Driven):")
display(factor_per_city)

In [ ]:
plt.figure(figsize=(10, 6))
bars = plt.barh(factor_per_city.index, factor_per_city['median'], color='skyblue')
plt.axvline(1.0, color='red', linestyle='--', label='Baseline Jabodetabek (1.0)')
plt.title('Regional Adjustment Factor: Jawa Tengah vs Jabodetabek')
plt.xlabel('Faktor Kalibrasi Harga')
plt.ylabel('Kota')
plt.legend()

for bar in bars:
    plt.text(bar.get_width()+0.01, bar.get_y() + bar.get_height()/2, 
             f'{bar.get_width():.2f}', va='center', fontweight='bold')

plt.show()

## 💡 Kesimpulan
Berdasarkan analisis *data-driven* ini, kita bisa memperbarui `REGIONAL_FACTORS` di aplikasi utama (`app.py`) dengan nilai riil yang didapatkan dari data pasar. Nilai ini menggantikan asumsi kasar sebelumnya.